In [101]:
SELECT * FROM 'Online Retail(3).csv';

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00+00:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00+00:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00+00:00,0.85,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00+00:00,2.10,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00+00:00,4.15,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00+00:00,4.15,12680,France


In [120]:
# Import SparkSession — the entry point for all PySpark functionality.
# Required to create a Spark application and interact with the Spark cluster.
from pyspark.sql import SparkSession  # add this import

# Initialize (or retrieve an existing) SparkSession with custom configuration.
# SparkSession.builder uses a fluent/chained API to set options before creating the
spark = (
    SparkSession.builder
    .appName("DataCamp PySpark Tutorial") # sets the application name, visible in the Spark UI
    .config("spark.memory.offHeap.enabled", "true") # enables off-heap memory storage,
                                                    # reducing JVM garbage collection pressure
    .config("spark.memory.offHeap.size", "10g")  # allocates 10GB of off-heap memory,
                                                 # useful for large datasets that exceed
                                                # JVM heap limits
    .getOrCreate()                              # returns existing session if one already exists,
                                # otherwise creates a new one — safe to call
                                                 # multiple times in the same notebook   
)


In [104]:
# Read the CSV file with multiLine=True to handle quoted newlines
df_spark = spark.read.csv("Online Retail(3).csv", header=True, escape='"', inferSchema=True, multiLine=True)
df_spark.show(5, 0)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850     |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
+---------+-----

In [69]:
# Display the first 5 rows of the DataFrame as a Pandas-style output.
# Useful for a quick visual sanity check of the data structure,
# column names, and sample values after loading.
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00+00:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00+00:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00+00:00,3.39,17850,United Kingdom


In [70]:
df.count()  # Answer: 2,500

InvoiceNo      541909
StockCode      541909
Description    540455
Quantity       541909
InvoiceDate    541909
UnitPrice      541909
CustomerID     406829
Country        541909
dtype: int64

In [71]:
df_spark.select('CustomerID').distinct().count() # Answer: 95

4373

In [72]:
# Wildcard imports bringing in all PySpark SQL functions (e.g. col, count, sum, max)
# and all PySpark data types (e.g. StringType, IntegerType, TimestampType).
# Convenient for exploratory work but not recommended in production as it can
# cause namespace collisions with Python built-ins like sum, max, and min.
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Group the dataset by Country and count the number of unique customers per country.
# count_distinct() is the wildcard-imported alias for countDistinct() —
# both are valid in PySpark but count_distinct() follows the newer snake_case convention.
# The result is aliased as 'country_count' for readability.
# Note: results are unordered — add .orderBy(desc('country_count')) to rank by size.
df_spark.groupBy('Country').agg(count_distinct('CustomerID').alias('country_count')).show()

+------------------+-------------+
|           Country|country_count|
+------------------+-------------+
|            Sweden|            8|
|         Singapore|            1|
|           Germany|           95|
|               RSA|            1|
|            France|           87|
|            Greece|            4|
|European Community|            1|
|           Belgium|           25|
|           Finland|           12|
|             Malta|            2|
|       Unspecified|            4|
|             Italy|           15|
|              EIRE|            3|
|         Lithuania|            1|
|            Norway|           10|
|             Spain|           31|
|           Denmark|            9|
|         Hong Kong|            0|
|            Israel|            4|
|           Iceland|            1|
+------------------+-------------+
only showing top 20 rows


In [121]:
# Import countDistinct for counting unique values and desc for descending ordering.
from pyspark.sql.functions import countDistinct, desc

# Group the dataset by Country and count the number of unique customers in each country.
# The result is aliased as 'country_count' and sorted in descending order,
# giving a ranked view of which countries have the most customers.
# Useful for understanding the geographic distribution of the customer base
# before filtering or segmenting the data for RFM analysis.
df_spark.groupBy('Country').agg(countDistinct('CustomerID').alias('country_count')).orderBy(desc('country_count')).show()

+---------------+-------------+
|        Country|country_count|
+---------------+-------------+
| United Kingdom|         3950|
|        Germany|           95|
|         France|           87|
|          Spain|           31|
|        Belgium|           25|
|    Switzerland|           21|
|       Portugal|           19|
|          Italy|           15|
|        Finland|           12|
|        Austria|           11|
|         Norway|           10|
|        Denmark|            9|
|Channel Islands|            9|
|      Australia|            9|
|    Netherlands|            9|
|         Sweden|            8|
|         Cyprus|            8|
|          Japan|            8|
|         Poland|            6|
|         Greece|            4|
+---------------+-------------+
only showing top 20 rows


In [123]:
import pyspark.sql.functions as F

# Parse the 'InvoiceDate' string column into a proper timestamp and store it as 'date'.
# coalesce() tries each format left to right, returning the first non-null result —
# making the parsing robust against multiple date formats present in the raw data.
df = df_spark.withColumn(
    "date",
    F.coalesce(
        F.to_timestamp(F.col("InvoiceDate"), "M/d/yy H:mm"),         # e.g. 12/1/10 8:26 (UCI format)
        F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss"),  # e.g. 2011-01-05 08:26:00
        F.to_timestamp(F.col("InvoiceDate"))                           # best-effort fallback
    )
)

# Display the most recent (maximum) date in the newly created 'date' column.
df.select(F.max("date")).show()

+-------------------+
|          max(date)|
+-------------------+
|2011-12-09 12:50:00|
+-------------------+



In [124]:
import pyspark.sql.functions as F

# Select and display the earliest (minimum) date value in the 'date' column.
# Useful for identifying the start boundary of the dataset's time range,
# which is used as the 'from_date' anchor point when calculating recency in the RFM model.
df.select(F.min("date")).show()

+-------------------+
|          min(date)|
+-------------------+
|2010-12-01 08:26:00|
+-------------------+



In [114]:
# Display the first 5 rows of the DataFrame with truncation disabled (0 = False),
# showing the full content of each cell without cutting off long values.
# Used as a quick sanity check to inspect raw data after loading or transformation.
df_spark.show(5,0)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |2010-12-01 08:26:00|2.55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |2010-12-01 08:26:00|2.75     |17850     |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |2010-12-01 08:26:00|3.39     |17850     |United Kingdom|
+---------+-----

In [126]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Always recompute 'date' as a proper timestamp to guarantee correct type,
# regardless of whether the column already exists on df.
df = df.withColumn(
    "date",
    F.coalesce(
        F.to_timestamp(F.col("InvoiceDate"), "M/d/yy H:mm"),         # e.g. 12/1/10 8:26 (UCI format)
        F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss"),  # e.g. 2011-01-05 08:26:00
        F.to_timestamp(F.col("InvoiceDate"))                           # best-effort fallback
    )
)

# Set the recency anchor date — the earliest transaction date in the dataset
df = df.withColumn("from_date", F.lit("2010-12-01 08:26:00").cast("timestamp"))

# Compute recency in seconds: larger value = more recent transaction
df2 = df.withColumn("recency", F.col("date").cast("long") - F.col("from_date").cast("long"))

# Use a window function to keep only the most recent transaction per customer
w = Window.partitionBy("CustomerID").orderBy(F.desc("recency"))
df2 = df2.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")

df2.show(5, truncate=False)

+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+-------------------+-------------------+--------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |date               |from_date          |recency |
+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+-------------------+-------------------+--------+
|581498   |15056bl  |EDWARDIAN PARASOL BLACK       |2       |2011-12-09 10:26:00|12.46    |NULL      |United Kingdom|2011-12-09 10:26:00|2010-12-01 08:26:00|32234400|
|C541433  |23166    |MEDIUM CERAMIC TOP STORAGE JAR|-74215  |2011-01-18 10:17:00|1.04     |12346     |United Kingdom|2011-01-18 10:17:00|2010-12-01 08:26:00|4153860 |
|581180   |23497    |CLASSIC CHROME BICYCLE BELL   |12      |2011-12-07 15:52:00|1.45     |12347     |Iceland       |2011-12-07 15:52:00|2010-12-01 08:26:00|32081160

In [127]:
# Group by CustomerID and find each customer's most recent transaction (max recency),
# then use a left semi join to keep only the rows in df2 where the recency value
# matches that customer's maximum — effectively retaining one row per customer
# with their latest transaction, without duplicating or adding new columns
df2 = df2.join(df2.groupBy('CustomerID').agg(max('recency').alias('recency')),on='recency',how='leftsemi')

In [128]:
# Import timestamp conversion functions from PySpark
from pyspark.sql.functions import to_timestamp, coalesce, col

# Overwrite the 'InvoiceDate' column with a properly parsed timestamp.
# coalesce() tries each format left to right and returns the first non-null result,
# making this robust against mixed or inconsistent date formats in the raw data.
df2_spark = df2.withColumn(
    "InvoiceDate",
    coalesce(
        to_timestamp(col("InvoiceDate"), "M/d/yy HH:mm"),
        to_timestamp(col("InvoiceDate"), "MM/dd/yyyy HH:mm"),
        to_timestamp(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("InvoiceDate"))  # generic fallback
    )
)

df2_spark.show(5, truncate=False)

+--------+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+-------------------+-------------------+
|recency |InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |date               |from_date          |
+--------+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+-------------------+-------------------+
|32234400|581498   |15056bl  |EDWARDIAN PARASOL BLACK       |2       |2011-12-09 10:26:00|12.46    |NULL      |United Kingdom|2011-12-09 10:26:00|2010-12-01 08:26:00|
|4153860 |C541433  |23166    |MEDIUM CERAMIC TOP STORAGE JAR|-74215  |2011-01-18 10:17:00|1.04     |12346     |United Kingdom|2011-01-18 10:17:00|2010-12-01 08:26:00|
|32081160|581180   |23497    |CLASSIC CHROME BICYCLE BELL   |12      |2011-12-07 15:52:00|1.45     |12347     |Iceland       |2011-12-07 15:52:00|2010-12-01 08:26:00

In [129]:
# Print the schema of df2 to verify column names, data types, and nullability.
# Useful for confirming that 'InvoiceDate' was correctly cast to TimestampType
# and that all other columns are the expected types before further transformations.
df2.printSchema()

root
 |-- recency: long (nullable = true)
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- from_date: timestamp (nullable = true)



In [130]:
# Group the DataFrame by CustomerID and count the number of invoices per customer.
# Each row in df2 represents a single transaction, so counting InvoiceDate entries
# per CustomerID gives us the purchase frequency — the 'F' in the RFM model.
# The result is stored in a new DataFrame df_freq with columns: CustomerID, frequency.
df_freq = df2.groupBy('CustomerID').agg(count('InvoiceDate').alias('frequency'))

In [131]:
# Display the first 5 rows of df_freq (the frequency DataFrame).
# The second argument 0 disables column truncation, ensuring full values are visible.
# Useful for verifying that the frequency aggregation per CustomerID looks correct.
df_freq.show(5,0)

+----------+---------+
|CustomerID|frequency|
+----------+---------+
|12346     |1        |
|12347     |1        |
|12348     |1        |
|12349     |1        |
|12350     |1        |
+----------+---------+
only showing top 5 rows


In [132]:
# Inner join df2 (which contains recency data) with df_freq (which contains frequency data)
# on CustomerID, combining the 'R' and 'F' components of the RFM model into one DataFrame.
# An inner join ensures only customers present in both DataFrames are retained,
# dropping any customers missing either a recency or frequency value.
df3 = df2.join(df_freq,on='CustomerID',how='inner')

In [133]:
# Print the schema of df3 to verify that the inner join between df2 and df_freq
# produced the expected columns (CustomerID, recency, frequency) with correct
# data types before proceeding to join with monetary value and build the final RFM table.
df3.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- recency: long (nullable = true)
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- from_date: timestamp (nullable = true)
 |-- frequency: long (nullable = false)



In [134]:
# Create a new column 'TotalAmount' representing the revenue generated per transaction,
# calculated as Quantity multiplied by UnitPrice.
# Both columns are explicitly cast to double to ensure accurate decimal arithmetic
# and to avoid integer truncation or type mismatch errors during multiplication.
# This forms the basis for aggregating the 'M' (monetary value) component of the RFM model.
m_val = df3.withColumn(
    "TotalAmount",
    col("Quantity").cast("double") * col("UnitPrice").cast("double")
)

In [135]:
# Aggregate m_val by CustomerID, summing all TotalAmount values per customer
# to get their total spend across all transactions.
# The result is aliased as 'monetary_value' — the 'M' in the RFM model —
# producing one row per customer with their cumulative purchase value.
m_val = m_val.groupBy('CustomerID').agg(sum('TotalAmount').alias('monetary_value'))

In [138]:
# Inner join the monetary value DataFrame (m_val) with df3 (which holds recency
# and frequency metrics) on CustomerID.
# An inner join keeps only customers that appear in both DataFrames,
# ensuring the final RFM table has no missing values across any of the three metrics.
finaldf = m_val.join(df3,on='CustomerID',how='inner')

In [137]:
# Select only the four RFM columns and CustomerID, dropping all other columns
# that are no longer needed for modelling (e.g. InvoiceDate, Description, etc.).
# .distinct() removes any duplicate rows that may have arisen from earlier
# joins or aggregations, ensuring one clean row per customer before scaling.
finaldf = finaldf.select(['recency','frequency','monetary_value','CustomerID']).distinct()

In [136]:
# Import VectorAssembler for combining columns into a single feature vector,
# and StandardScaler for normalizing the feature values.
from pyspark.ml.feature import VectorAssembler, StandardScaler

# VectorAssembler combines the three RFM columns into a single 'features' vector column.
# This is required by PySpark ML — all input features must be in one vector before
# any ML algorithm or scaler can be applied.
assemble = VectorAssembler(
    inputCols=["recency", "frequency", "monetary_value"],
    outputCol="features"
)
# Apply the assembler to finaldf, producing a new 'features' column on each row
assembled_data = assemble.transform(finaldf)

# Define a StandardScaler to normalize the 'features' vector.
# Standardization ensures no single RFM metric dominates due to differences in scale
# (e.g. monetary_value in the hundreds vs frequency in single digits).
scale = StandardScaler(inputCol="features", outputCol="standardized")
# Fit the scaler on assembled_data to compute the mean and standard deviation
# of each feature across the dataset — this does not transform the data yet.
data_scale = scale.fit(assembled_data)
# Apply the fitted scaler to assembled_data, producing a new 'standardized' column
# containing the normalized feature vectors — ready for clustering or modelling.
data_scale_output = data_scale.transform(assembled_data)


In [86]:
# Display the first 2 rows of the 'standardized' column from data_scale_output without truncating the output
data_scale_output.select('standardized').show(2,truncate=False)

+----------------------------------------------+
|standardized                                  |
+----------------------------------------------+
|[0.4770929272664938,0.0,-27.474245208216594]  |
|[3.6846919574816557,0.0,0.0061936974515696165]|
+----------------------------------------------+
only showing top 2 rows
